In [ ]:
import os
import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dataset = 'HCP-YA'
data_loader = DataLoader(dataset=dataset)

# MEG data

In [ ]:
subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_meg.txt'), dtype=str)
subject_ids

In [ ]:
meg_scans = data_loader.register_meg_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
    rest_runs=[3, 4, 5],
    wm_runs=[6, 7]
)

In [ ]:
meg_scans = data_loader.validate_scan_registry()

In [ ]:
data_loader.export_scan_registry('scan_registry_meg.csv')
scan_registry = data_loader.list_registered_scans()
scan_registry.head()

In [ ]:
subject_ids = scan_registry['subject_id'].unique()
subject_ids

## Process subjects one at a time (memory issues)

In [ ]:
freq_windows = [
    None,
    (0, 6),
    (0, 7)
]

timescale_measures = [
    'zero_crossing',
    'auc_zero_crossing'
]

In [ ]:
# run_task = False

for run_task in [False, True]:
    if run_task:
        rest_runs = None
        wm_runs = [6, 7]
    else:
        rest_runs = [3, 4, 5]
        wm_runs = None

    for subject_id in subject_ids:
        data_loader = DataLoader(dataset=dataset)

        meg_scans = data_loader.register_meg_scans(
            subject_ids=[subject_id,],
            atlas='Schaefer4007',
            rest_runs=rest_runs,
            wm_runs=wm_runs
        )

        meg_data = data_loader.load_batch(
            modalities=['MEG'],
            nuisance_regression=False,
            verbose=True
        )
        
        timescales_processor = Timescales(dataset=dataset, timeseries_data=meg_data, modality='MEG')
        if run_task:
            timescales_processor.regress_task_structure()
            
        for freq_window in freq_windows:
            timescales_processor.compute_autocorrelation_function(use_fft=True, freq_window=freq_window, use_residuals=False)
            for timescale_measure in timescale_measures:
                timescales_processor.compute_timescale_measures(method=timescale_measure, suffix=f'freq-{freq_window}')
            
            if run_task:
                timescales_processor.compute_autocorrelation_function(use_fft=True, freq_window=freq_window, use_residuals=True)
                for timescale_measure in timescale_measures:
                    timescales_processor.compute_timescale_measures(method=timescale_measure, suffix=f'resid_freq-{freq_window}')